# Supplementary Figure 1 — extended benchmark and recovery detail

Each panel loads its own precomputed CSV output and builds its plot
natively in this notebook (not a rendered PNG from another notebook), so
panels compose at a consistent scale and font size without raster
stretching — same convention as `fig2.ipynb`.

💡 **Environment:** `clamp-analyses`

## Setup

In [ ]:
suppressPackageStartupMessages({
  library(data.table)
  library(ggplot2)
  library(patchwork)   # all layout/composition
  library(cowplot)     # kept for its grob helpers; not used for layout
  library(yaml)
  library(here)
  library(grid)        # textGrob()/gpar() for panel A's shared axis title
  library(ragg)        # PNG device (better AA + font rendering than png())
  library(svglite)     # SVG device
})


In [ ]:
# ============================================================
# Central typography configuration (Nature Methods)
# ------------------------------------------------------------
# Nature Methods: sans-serif throughout, body text 5-7pt, panel
# tags bold and >= 8pt. Every text size in this notebook is one
# of these constants -- no per-panel magic numbers.
# ============================================================
FONT_FAMILY <- "Helvetica"

FS_TAG          <- 9      # panel tags (bold)
FS_TITLE        <- 8      # panel titles (bold)
FS_SUBTITLE     <- 6.5    # panel subtitles
FS_AXIS_TITLE   <- 7
FS_AXIS_TEXT    <- 6.5
FS_LEGEND       <- 5.5
FS_LEGEND_TITLE <- 6
FS_STAT         <- 6      # statistical annotations (brackets)
FS_HEAT_LABEL   <- 5.5    # heatmap row/column labels
FS_HEAT_VALUE   <- 5.5    # standard heatmap cell values (panel E)
FS_CELL_VALUE   <- 5      # standard compact numeric annotations
FS_DENSE_VALUE  <- 4      # dense D-row-1 and H numeric annotations
FS_A_MEAN       <- 4      # three-decimal means above panel A boxplots
# Panel H alone is a 49 x 49 matrix; at one A4 page its 98 axis labels do
# not fit at 5.5 pt. 5 pt is Nature's stated floor for figure text and is
# used there, and nowhere else.
FS_HEAT_LABEL_H <- 5

LINE_W <- 0.3             # axis / tick / tile-border line width
GRID_W <- 0.2             # reference grid lines

# theme sizes are pt, geom_text()/annotate() sizes are mm
pt_mm <- function(pt) pt / .pt

# ============================================================
# Reusable theme
# ============================================================
theme_nature_methods <- function(base_size = FS_AXIS_TEXT,
                                 grid = c("none", "y", "x", "both")) {
  grid <- match.arg(grid)
  th <- theme_classic(base_size = base_size, base_family = FONT_FAMILY) %+replace%
    theme(
      plot.title        = element_text(size = FS_TITLE, face = "bold", hjust = 0.5,
                                       margin = margin(b = 1)),
      plot.subtitle     = element_text(size = FS_SUBTITLE, face = "plain", hjust = 0.5,
                                       lineheight = 0.95, margin = margin(b = 1)),
      plot.tag          = element_text(size = FS_TAG, face = "bold", family = FONT_FAMILY),
      plot.tag.position = "topleft",
      plot.tag.location = "margin",
      axis.line         = element_line(linewidth = LINE_W, colour = "black"),
      axis.ticks        = element_line(linewidth = LINE_W, colour = "black"),
      axis.ticks.length = unit(0.9, "pt"),
      axis.text         = element_text(size = base_size, colour = "black"),
      axis.text.x       = element_text(margin = margin(t = 0.8)),
      axis.text.y       = element_text(hjust = 1, margin = margin(r = 0.8)),
      axis.title        = element_text(size = FS_AXIS_TITLE, colour = "black"),
      axis.title.x      = element_text(margin = margin(t = 1)),
      axis.title.y      = element_text(angle = 90, margin = margin(r = 1)),
      legend.text       = element_text(size = FS_LEGEND),
      legend.title      = element_text(size = FS_LEGEND_TITLE),
      legend.key.size   = unit(2, "mm"),
      legend.key        = element_blank(),
      legend.background = element_blank(),
      legend.margin     = margin(0, 0, 0, 0),
      legend.box.margin = margin(0, 0, 0, 0),
      strip.text        = element_text(size = FS_TITLE, face = "bold", margin = margin(b = 1)),
      strip.background  = element_blank(),
      panel.background  = element_blank(),
      panel.grid        = element_blank(),
      plot.background   = element_blank(),
      plot.margin       = margin(1, 1, 1, 1, "mm")
    )
  if (grid %in% c("y", "both"))
    th <- th + theme(panel.grid.major.y = element_line(colour = "grey88", linewidth = GRID_W))
  if (grid %in% c("x", "both"))
    th <- th + theme(panel.grid.major.x = element_line(colour = "grey88", linewidth = GRID_W))
  th
}

# Heatmap variant: no axis line/ticks, smaller category labels.
theme_nature_heatmap <- function(x_angle = 45, label_size = FS_HEAT_LABEL) {
  theme_nature_methods() %+replace%
    theme(
      axis.line   = element_blank(),
      axis.ticks  = element_blank(),
      axis.text.x = element_text(size = label_size, angle = x_angle,
                                 hjust = if (x_angle == 0) 0.5 else 1,
                                 vjust = if (x_angle == 90) 0.5 else 1,
                                 colour = "black",
                                 margin = margin(t = 0.5), lineheight = 0.9),
      axis.text.y = element_text(size = label_size, hjust = 1, colour = "black",
                                 margin = margin(r = 0.5), lineheight = 0.9),
      plot.margin = margin(1, 1, 1, 1, "mm")
    )
}

# One tag per major block: applied to the block's top-left leaf so the
# letter lands at the block's own upper-left corner. The tag goes in the
# plot's margin slot ("topleft"), which reserves its own space -- a
# coordinate position instead draws it over the y-axis title.
add_tag <- function(p, tag) {
  p + labs(tag = tag) +
    theme(plot.tag          = element_text(size = FS_TAG, face = "bold",
                                           family = FONT_FAMILY),
          plot.tag.position = "topleft",
          plot.tag.location = "margin")
}

# Overlay tag for blocks where a margin tag would reserve a wide gtable
# column and push the y-axis title away from the data. cowplot's drawing
# canvas contributes no width or height to the patchwork layout, and the
# opaque wrapper prevents neighbouring plots from borrowing its axis columns.
add_overlay_tag <- function(p, tag) {
  tagged <- ggdraw(p) +
    draw_label(tag, x = 0, y = 1, hjust = 0, vjust = 1,
               size = FS_TAG, fontface = "bold", fontfamily = FONT_FAMILY)
  wrap_elements(full = tagged)
}

# ============================================================
# Labels, palettes, helpers
# ============================================================
DATASET_LABELS <- c(
  Brain_Mathys2023 = "Brain Mathys",
  Brain_Xiong2023  = "Brain Xiong",
  Heart_Datar2026  = "Heart Datar",
  PBMC_1k1k        = "PBMC 1k1k",
  PBMC_Perez2022   = "PBMC Perez",
  Lung_Sikkema2023 = "Lung Sikkema"
)

TISSUE_MAP <- c(
  Brain_Mathys2023 = "Brain", Brain_Xiong2023 = "Brain",
  Heart_Datar2026  = "Heart",
  Lung_Sikkema2023 = "Lung",
  PBMC_1k1k        = "PBMC", PBMC_Perez2022 = "PBMC"
)

# Explicit, documented dataset placement for the six-heatmap blocks (C and
# D). Fixed by hand -- NOT sorted by category count -- so C and D show the
# same dataset in the same grid position and the reader can compare them.
DATASETS_ROW1 <- c("Heart_Datar2026", "PBMC_1k1k", "Lung_Sikkema2023")
DATASETS_ROW2 <- c("Brain_Mathys2023", "Brain_Xiong2023", "PBMC_Perez2022")

# The six datasets have 4-14 categories. patchwork's numeric widths and
# heights size the panel (null) areas, with axis labels and titles added on
# top as fixed extras, so passing the raw category counts as widths/heights
# is what makes one tile the same size in every dataset. See panels C and D.

TISSUE_ORDER <- c("Brain", "Heart", "Lung", "PBMC")

abbreviate_ct <- function(x) {
  x <- gsub("Oligodendrocyte [Pp]rogenitor [Cc]ells?", "OPC", x)
  x <- gsub("Oligodendrocyte [Pp]recursor [Cc]ells?", "OPC", x)
  x
}

# Compact cell-type labels (<= ~12 characters) for the heatmap axes. Long
# names are what previously forced a ~18 mm band of rotated text under
# every heatmap and squeezed the tiles down to nothing. Standard
# abbreviations only; spelled out in the figure caption.
CT_SHORT <- c(
  "Oligodendrocyte Precursor Cells" = "OPC",
  "Plasmacytoid Dendritic Cells"    = "pDC",
  "LymphaticEndothelial"            = "Lymphatic EC",
  "Alveolar epithelium"             = "Alveolar ep.",
  "Airway epithelium"               = "Airway ep.",
  "Excitatory neurons"              = "Excitatory",
  "Inhibitory neurons"              = "Inhibitory",
  "Endothelial cells"               = "Endothelial",
  "Fibroblast lineage"              = "Fibroblast",
  "Submucosal Gland"                = "Submucosal",
  "Oligodendrocytes"                = "Oligodendro.",
  "CD14+ Monocytes"                 = "CD14+ Mono",
  "CD16+ Monocytes"                 = "CD16+ Mono",
  "Dendritic cells"                 = "Dendritic",
  "Plasma B cells"                  = "Plasma B",
  "Myeloid cells"                   = "Myeloid",
  "Malignant cells"                 = "Malignant",
  "Vascular cells"                  = "Vascular",
  "Cancer cells"                    = "Cancer",
  "Mast cells"                      = "Mast",
  "Blood vessels"                   = "Blood vessel"
)
short_ct <- function(x) {
  x <- abbreviate_ct(x)
  ifelse(x %in% names(CT_SHORT), CT_SHORT[x], x)
}

wrap_label <- function(x, width = 14) {
  vapply(x, function(s) paste(strwrap(s, width = width), collapse = "\n"),
         character(1), USE.NAMES = FALSE)
}

# Compact correlation values without dropping precision: .95 instead of
# 0.95 and -.73 instead of -0.73. This keeps 5 pt text inside small cells.
fmt_corr_cell <- function(x) sub("0.", ".", sprintf("%.2f", x), fixed = TRUE)

# Compact GTEx subtissue labels for panel H (49 rows x 49 columns). Only
# redundant/parenthetical wording is removed -- every subtissue stays
# distinguishable, and the abbreviations are listed in the figure caption.
shorten_gtex <- function(x) {
  x <- gsub("Adipose - Subcutaneous",                    "Adipose - Subcut.", x)
  x <- gsub("Adipose - Visceral \\(Omentum\\)",             "Adipose - Visceral", x)
  x <- gsub("Artery - ",                                 "Artery - ", x)
  x <- gsub("Brain - Amygdala",                          "Brain - Amygdala", x)
  x <- gsub("Brain - Anterior cingulate cortex \\(BA24\\)", "Brain - ACC (BA24)", x)
  x <- gsub("Brain - Caudate \\(basal ganglia\\)",         "Brain - Caudate", x)
  x <- gsub("Brain - Cerebellar Hemisphere",             "Brain - Cereb. hem.", x)
  x <- gsub("Brain - Frontal Cortex \\(BA9\\)",            "Brain - FC (BA9)", x)
  x <- gsub("Brain - Nucleus accumbens \\(basal ganglia\\)", "Brain - NAc", x)
  x <- gsub("Brain - Putamen \\(basal ganglia\\)",         "Brain - Putamen", x)
  x <- gsub("Brain - Spinal cord \\(cervical c-1\\)",      "Brain - Spinal cord", x)
  x <- gsub("Brain - Substantia nigra",                  "Brain - Subst. nigra", x)
  x <- gsub("Breast - Mammary Tissue",                   "Breast - Mammary", x)
  x <- gsub("Cells - Cultured fibroblasts",              "Cells - Fibroblasts", x)
  x <- gsub("Cells - EBV-transformed lymphocytes",       "Cells - EBV lymph.", x)
  x <- gsub("Esophagus - Gastroesophageal Junction",     "Esoph. - GE junction", x)
  x <- gsub("Esophagus - Mucosa",                        "Esoph. - Mucosa", x)
  x <- gsub("Esophagus - Muscularis",                    "Esoph. - Muscularis", x)
  x <- gsub("Heart - Atrial Appendage",                  "Heart - Atrial app.", x)
  x <- gsub("Minor Salivary Gland",                      "Minor saliv. gland", x)
  x <- gsub("Skin - Not Sun Exposed \\(Suprapubic\\)",     "Skin - Not sun exp.", x)
  x <- gsub("Skin - Sun Exposed \\(Lower leg\\)",          "Skin - Sun exp.", x)
  x <- gsub("Small Intestine - Terminal Ileum",          "Sm. intestine - Ileum", x)
  trimws(x)
}

# Compact adjusted-P formatting for the F/G significance brackets
# (parsed as a plotmath expression, so 10^-8 sets as a real superscript).
fmt_q_compact <- function(q) {
  if (is.na(q)) return('"n.a."')
  if (q >= 0.001) return(sprintf('"%.3f"', q))
  e_str    <- formatC(q, format = "e", digits = 1)
  parts    <- strsplit(e_str, "e")[[1]]
  sprintf('%s%%*%%10^{%d}', trimws(parts[1]), as.integer(parts[2]))
}

# Greedy tier assignment so significance brackets never overlap.
assign_bracket_tiers <- function(comp_df, xpos) {
  comp_ord <- copy(as.data.table(comp_df))
  comp_ord[, x1 := xpos[a]]
  comp_ord[, x2 := xpos[b]]
  comp_ord[, left  := pmin(x1, x2)]
  comp_ord[, right := pmax(x1, x2)]
  comp_ord[, span  := right - left]
  setorder(comp_ord, span, q)

  levels_used <- list()
  comp_ord[, tier := 0L]
  for (i in seq_len(nrow(comp_ord))) {
    left  <- comp_ord$left[i]; right <- comp_ord$right[i]; tier <- 1L
    repeat {
      current  <- if (tier <= length(levels_used)) levels_used[[tier]] else NULL
      overlaps <- !is.null(current) && any(vapply(current, function(iv) {
        !(right < iv[1] || left > iv[2])
      }, logical(1)))
      if (!overlaps) break
      tier <- tier + 1L
    }
    comp_ord$tier[i] <- tier
    prior <- if (tier <= length(levels_used)) levels_used[[tier]] else list()
    levels_used[[tier]] <- c(prior, list(c(left, right)))
  }
  comp_ord
}

# Draw a set of tiered significance brackets onto a boxplot panel.
add_brackets <- function(p, comp_ord, y_base, y_step, h, size = pt_mm(FS_STAT)) {
  for (i in seq_len(nrow(comp_ord))) {
    r <- comp_ord[i, ]
    y <- y_base + (r$tier - 1) * y_step
    p <- p +
      annotate("segment", x = r$left,  xend = r$left,  y = y,     yend = y + h, linewidth = 0.2) +
      annotate("segment", x = r$left,  xend = r$right, y = y + h, yend = y + h, linewidth = 0.2) +
      annotate("segment", x = r$right, xend = r$right, y = y,     yend = y + h, linewidth = 0.2) +
      annotate("text", x = (r$left + r$right) / 2, y = y + h + 0.005,
               label = fmt_q_compact(r$q), size = size, vjust = 0,
               parse = TRUE, family = FONT_FAMILY)
  }
  p
}

cfg <- yaml::read_yaml(here("config.yaml"))
MODEL_COLORS_RAW <- unlist(cfg$MODEL_COLORS)
names(MODEL_COLORS_RAW)[names(MODEL_COLORS_RAW) == "GenomicSuperSignature"] <- "GSSig"

ct_labels_df <- read.csv(here("data", "pseudobulk", "cell_type_labels.csv"), stringsAsFactors = FALSE)
CT_LABELS <- setNames(ct_labels_df$label, ct_labels_df$cell_type)
ct_label <- function(x) ifelse(x %in% names(CT_LABELS), CT_LABELS[x], x)

GREEN_SCALE      <- unlist(cfg$GREEN_SCALE)
DIVERGING_COLORS <- unlist(cfg$DIVERGING_COLORS)
DIVERGING_VALUES <- as.numeric(cfg$DIVERGING_VALUES)

# Text colour for values printed on top of a filled tile.
TILE_TEXT <- c(`TRUE` = "white", `FALSE` = "black")


## Panel A: benchmark boxplots, faceted by tissue

In [ ]:
long <- fread(snakemake@input[["benchmark_long"]])
stopifnot(all(c("dataset", "method", "truth", "cor") %in% names(long)))

long_box <- long[truth == "v0" & !is.na(cor)]
long_box[, tissue := TISSUE_MAP[dataset]]
stopifnot(!anyNA(long_box$tissue))

# One method order, defined once, reused by every tissue plot (and by
# panel B): descending overall mean max-r.
METHOD_ORDER  <- long_box[, .(m = mean(cor, na.rm = TRUE)), by = method][order(-m), as.character(method)]
METHOD_COLORS <- MODEL_COLORS_RAW[METHOD_ORDER]
METHOD_COLORS[is.na(METHOD_COLORS)] <- "grey70"
names(METHOD_COLORS) <- METHOD_ORDER

long_box[, method := factor(method, levels = METHOD_ORDER)]
mean_by_tissue <- long_box[, .(mean_cor = mean(cor, na.rm = TRUE),
                               max_cor  = max(cor, na.rm = TRUE)), by = .(tissue, method)]

# Headroom above y = 1 for the per-method mean labels.
A_Y_MAX <- 1.16

# One tissue = one plot. Repeated tick labels are switched off per position;
# patchwork collects the identical y-axis titles into one title for the grid.
make_tissue_box <- function(tis, show_x, show_y) {
  d <- long_box[tissue == tis]
  m <- mean_by_tissue[tissue == tis]
  p <- ggplot(d, aes(method, cor, fill = method)) +
    geom_boxplot(width = 0.6, outlier.shape = NA, colour = "black",
                 linewidth = LINE_W, alpha = 0.85) +
    geom_jitter(width = 0.10, size = 0.3, shape = 21, fill = "white",
                colour = "#333333", stroke = 0.12, alpha = 0.7) +
    geom_point(data = m, aes(x = method, y = mean_cor), shape = 23, size = 1.1,
               fill = "white", colour = "black", stroke = 0.3, inherit.aes = FALSE) +
    geom_text(data = m, aes(x = method, y = 1.05, label = sprintf("%.3f", mean_cor)),
              size = pt_mm(FS_A_MEAN), colour = "black", inherit.aes = FALSE) +
    scale_x_discrete(drop = FALSE) +
    scale_y_continuous(breaks = seq(0, 1, 0.25),
                       labels = c("0", "0.25", "0.5", "0.75", "1.0"),
                       expand = expansion(mult = c(0.02, 0.02))) +
    coord_cartesian(ylim = c(0, A_Y_MAX), clip = "on") +
    scale_fill_manual(values = METHOD_COLORS, na.value = "grey70", drop = FALSE) +
    labs(x = NULL, y = "Max Pearson r per cell type", title = tis) +
    theme_nature_methods(grid = "none") +
    theme(legend.position = "none")
  p <- if (show_x) {
    p + theme(axis.text.x = element_text(angle = 35, hjust = 1, vjust = 1,
                                         size = FS_AXIS_TEXT, margin = margin(t = 0.8)))
  } else {
    p + theme(axis.text.x = element_blank(), axis.ticks.x = element_blank())
  }
  if (!show_y) p <- p + theme(axis.text.y = element_blank(), axis.ticks.y = element_blank())
  p
}

plot_A_brain <- make_tissue_box("Brain", show_x = FALSE, show_y = TRUE)
plot_A_heart <- make_tissue_box("Heart", show_x = FALSE, show_y = FALSE)
plot_A_lung  <- make_tissue_box("Lung",  show_x = TRUE,  show_y = TRUE)
plot_A_pbmc  <- make_tissue_box("PBMC",  show_x = TRUE,  show_y = FALSE)

# Both rows receive the same panel height so all four tissue plots have the
# same plotting-area dimensions; rotated labels use the bottom margin.
# wrap_plots() rather than the `|` / `/` operators throughout: those two
# operators flatten an operand that is already a patchwork into the parent
# level, which silently breaks nested blocks. wrap_plots() always nests.
A_grid <- wrap_plots(plot_A_brain, plot_A_heart, plot_A_lung, plot_A_pbmc,
                     ncol = 2, heights = c(1, 1)) +
  plot_layout(axis_titles = "collect_y")

# The overlay tag adds no layout column, so the shared title sits directly
# beside the tick labels and all remaining width belongs to the boxplots.
panel_A <- add_overlay_tag(A_grid, "A")

options(repr.plot.width = 6, repr.plot.height = 3)
print(panel_A)


## Panel B: bootstrap win rate

In [ ]:
win_rate <- fread(snakemake@input[["bootstrap"]])
stopifnot(all(c("method", "win_rate") %in% names(win_rate)))

# Horizontal bars: the ten method names read left-to-right at 0 degrees and
# the bars use the full height that panel A sets for this row. Ascending
# factor order puts the best method at the top of a y-axis.
setorder(win_rate, win_rate)
win_rate[, method := factor(method, levels = method)]

panel_B <- ggplot(win_rate, aes(x = win_rate, y = method, fill = method)) +
  geom_col(width = 0.72) +
  geom_text(aes(label = sprintf("%.0f%%", 100 * win_rate)),
            hjust = -0.2, size = pt_mm(FS_AXIS_TEXT), colour = "black") +
  scale_fill_manual(values = MODEL_COLORS_RAW, na.value = "grey70") +
  scale_x_continuous(limits = c(0, 1), breaks = seq(0, 1, 0.25),
                     labels = function(x) paste0(100 * x, "%"),
                     expand = expansion(mult = c(0, 0.22))) +
  labs(x = NULL, y = NULL) +
  theme_nature_methods(grid = "none") +
  theme(legend.position = "none",
        axis.line.y   = element_blank(),
        axis.ticks.y  = element_blank(),
        axis.text.x   = element_text(margin = margin(t = 0)),
        axis.ticks.length.x = unit(0.6, "pt"),
        plot.margin   = margin(1, 1, 0.3, 1, "mm"))

panel_B <- add_tag(panel_B, "B")

options(repr.plot.width = 2, repr.plot.height = 3)
print(panel_B)


## Panel C: per-dataset single-cell recovery (purity) heatmaps

In [ ]:
heatmap_long <- fread(snakemake@input[["heatmap_long"]])
stopifnot(all(c("dataset", "row_cell_type", "col_cell_type", "pct") %in% names(heatmap_long)))

# Cell values are printed only where they carry information: the diagonal
# (the quantity the panel is about) and off-diagonal leakage >= 15%.
# Everything below that is read from the fill scale, which is what turned
# the previous version's dense grids into blocks of overlapping digits.
PURITY_LABEL_MIN <- 15

make_purity_panel <- function(ds) {
  d <- heatmap_long[dataset == ds]
  stopifnot(nrow(d) > 0)

  ct_order  <- unique(d$row_cell_type)
  ct_order  <- ct_order[order(ct_label(ct_order))]
  labs_ord  <- short_ct(ct_label(ct_order))

  # Cell-type names on both axes. They are compacted by short_ct() rather
  # than replaced by indices, so each axis reads on its own.
  d[, row_label   := factor(short_ct(ct_label(row_cell_type)), levels = labs_ord)]
  d[, col_label   := factor(short_ct(ct_label(col_cell_type)), levels = labs_ord)]
  d[, is_diagonal := row_cell_type == col_cell_type]
  d[, show_label  := is_diagonal | pct >= PURITY_LABEL_MIN]

  ggplot(d, aes(x = col_label, y = row_label, fill = pct)) +
    geom_tile(colour = "white", linewidth = 0.25) +
    geom_tile(data = d[is_diagonal == TRUE], fill = NA, colour = "black", linewidth = 0.4) +
    geom_text(data = d[show_label == TRUE],
              aes(label = sprintf("%.0f", pct), colour = pct >= 65),
              size = pt_mm(FS_CELL_VALUE), show.legend = FALSE) +
    scale_fill_gradientn(colours = GREEN_SCALE, limits = c(0, 100),
                         name = "Top 1% purity (%)",
                         guide = guide_colourbar(barwidth = unit(20, "mm"),
                                                 barheight = unit(1.8, "mm"),
                                                 title.position = "left",
                                                 title.vjust = 1)) +
    scale_colour_manual(values = TILE_TEXT, guide = "none") +
    scale_x_discrete(expand = c(0, 0)) +
    scale_y_discrete(expand = c(0, 0)) +
    labs(x = NULL, y = NULL, title = DATASET_LABELS[ds]) +
    theme_nature_heatmap(x_angle = 45, label_size = FS_HEAT_LABEL_H)
}

C_row1_panels <- lapply(DATASETS_ROW1, make_purity_panel)
C_row2_panels <- lapply(DATASETS_ROW2, make_purity_panel)

# Within each row, category-count weights give denser datasets more width.
# The two rows use independent scales so the smaller lower panels can grow.
C_N1 <- vapply(DATASETS_ROW1, function(ds) uniqueN(heatmap_long[dataset == ds, row_cell_type]), numeric(1))
C_N2 <- vapply(DATASETS_ROW2, function(ds) uniqueN(heatmap_long[dataset == ds, row_cell_type]), numeric(1))

# The bottom row uses the full available width instead of preserving the
# top row's tile scale. Its shared legend occupies a compact fourth slot.
C_LEG <- max(C_N2)

C_row1 <- wrap_plots(C_row1_panels, nrow = 1, widths = C_N1) &
  theme(legend.position = "none")
C_row2 <- wrap_plots(c(C_row2_panels, list(guide_area())),
                     nrow = 1, widths = c(C_N2, C_LEG),
                     guides = "collect") &
  theme(legend.position = "bottom", legend.direction = "horizontal",
        legend.justification = "left")

# One pair of short display labels describes the axes shared by all six
# heatmaps. Their full source-notebook wording is:
# x: Cell type whose assigned LV was used to rank cells
# y: Annotated cell type of the top 1% projected cells
panel_C_core <- wrap_plots(C_row1, C_row2, ncol = 1, heights = c(1.35, 1))
panel_C <- wrap_elements(full =
  ggdraw() +
    draw_plot(panel_C_core, x = 0.042, y = 0.038, width = 0.958, height = 0.962) +
    draw_label("LV assigned to cell type", x = 0.52, y = 0.002,
               hjust = 0.5, vjust = 0, size = FS_AXIS_TITLE,
               fontfamily = FONT_FAMILY) +
    draw_label("Cell type of top 1% projected cells", x = 0.006, y = 0.52,
               angle = 90, hjust = 0.5, vjust = 0, size = FS_AXIS_TITLE,
               fontfamily = FONT_FAMILY) +
    draw_label("C", x = 0, y = 1, hjust = 0, vjust = 1,
               size = FS_TAG, fontface = "bold", fontfamily = FONT_FAMILY)
)

options(repr.plot.width = 7.2, repr.plot.height = 2.5)
print(panel_C)


## Panel D: per-dataset LV x cell-type correlation heatmaps

In [ ]:
corr_full          <- fread(snakemake@input[["corr_full"]])
assignments        <- fread(snakemake@input[["assignments"]])
stopifnot(all(c("dataset", "LV", "cell_type", "cor") %in% names(corr_full)))
stopifnot(all(c("dataset", "LV", "cell_type") %in% names(assignments)))

# Printed values are restricted to the assigned LV <-> cell-type pair (the
# claim the panel makes) plus any other coefficient strong enough to matter
# (|r| >= 0.6). The rest is read from the diverging fill scale.
CORR_LABEL_MIN <- 0.6

make_corr_panel <- function(ds) {
  d        <- corr_full[dataset == ds & !is.na(cell_type) & cell_type != "NA"]
  assigned <- assignments[dataset == ds & !is.na(cell_type) & cell_type != "NA"]
  stopifnot(nrow(d) > 0, nrow(assigned) > 0)
  d <- d[LV %in% assigned$LV & cell_type %in% assigned$cell_type]

  ord      <- order(assigned$cell_type)
  lv_order <- unique(assigned$LV[ord])
  ct_order <- unique(short_ct(ct_label(assigned$cell_type[ord])))

  assigned_key <- paste(assigned$LV, assigned$cell_type)
  d[, cell_type_label := factor(short_ct(ct_label(cell_type)), levels = ct_order)]
  d[, LV              := factor(LV, levels = lv_order)]
  d[, is_assigned     := paste(LV, cell_type) %in% assigned_key]
  d[, show_label      := is_assigned | abs(cor) >= CORR_LABEL_MIN]
  value_size <- if (ds %in% DATASETS_ROW1) FS_DENSE_VALUE else FS_CELL_VALUE

  # Transposed relative to the raw table: LV on x (short labels, so the
  # rotated band under the panel stays a few millimetres) and cell type on
  # y, where the long names read horizontally and cost width, not height.
  ggplot(d, aes(x = LV, y = cell_type_label, fill = cor)) +
    geom_tile(colour = "white", linewidth = 0.25) +
    geom_tile(data = d[is_assigned == TRUE], fill = NA, colour = "black", linewidth = 0.4) +
    geom_text(data = d[show_label == TRUE],
              aes(label = fmt_corr_cell(cor), colour = abs(cor) >= 0.8),
              size = pt_mm(value_size), show.legend = FALSE) +
    scale_fill_gradientn(colours = DIVERGING_COLORS, values = DIVERGING_VALUES,
                         limits = c(-1, 1), name = "Pearson r",
                         guide = guide_colourbar(barwidth = unit(20, "mm"),
                                                 barheight = unit(1.8, "mm"),
                                                 title.position = "left",
                                                 title.vjust = 1)) +
    scale_colour_manual(values = TILE_TEXT, guide = "none") +
    scale_x_discrete(expand = c(0, 0)) +
    scale_y_discrete(expand = c(0, 0)) +
    labs(x = NULL, y = NULL, title = DATASET_LABELS[ds]) +
    theme_nature_heatmap(x_angle = 45, label_size = FS_HEAT_LABEL_H) +
    theme(plot.title    = element_text(size = FS_SUBTITLE + 0.5, face = "bold",
                                       hjust = 0.5, margin = margin(b = 0.5)),
          plot.subtitle = element_blank())
}

D_row1_panels <- lapply(DATASETS_ROW1, make_corr_panel)
D_row2_panels <- lapply(DATASETS_ROW2, make_corr_panel)
D_row1_panels[[1]] <- add_tag(D_row1_panels[[1]], "D")

# Within each row, category-count weights give denser datasets more width.
# The lower row fills the block independently instead of inheriting the
# upper row's smaller tile scale.
D_ncat <- function(ds) uniqueN(assignments[dataset == ds & !is.na(cell_type) & cell_type != "NA", LV])
D_N1 <- vapply(DATASETS_ROW1, D_ncat, numeric(1))
D_N2 <- vapply(DATASETS_ROW2, D_ncat, numeric(1))
D_row1 <- wrap_plots(D_row1_panels, nrow = 1, widths = D_N1)
D_row2 <- wrap_plots(D_row2_panels, nrow = 1, widths = D_N2)

# No guides = "collect" here: D and E use the same scale and share one
# legend, collected once at the row_DE level in the assembly cell.
panel_D <- wrap_plots(D_row1, D_row2, ncol = 1, heights = c(1.35, 1))

options(repr.plot.width = 5, repr.plot.height = 2.5)
print(panel_D)


## Panel E: difficult-to-distinguish cell-type pairs (own panel letter)

In [ ]:
related_corr <- fread(snakemake@input[["related_corr"]])
stopifnot(all(c("group_id", "LV", "cell_type", "assigned_cell_type", "cor") %in% names(related_corr)))

# Explicit comparison order (group_id 1-6 as produced upstream):
#   1 B cells vs T cells                4 astrocytes vs microglia
#   2 CD4+ vs CD8+ T cells              5 excitatory vs inhibitory neurons
#   3 OPC vs oligodendrocytes           6 atrial vs ventricular cardiomyocytes
E_GROUP_ORDER <- sort(unique(related_corr$group_id))

pair_axis_label <- function(x) {
  x <- short_ct(ct_label(x))
  x <- sub(" cells$", "", x)
  x <- sub("^VentricularCM$", "Ventric. CM", x)
  x <- sub("^AtrialCM$", "Atrial CM", x)
  x
}

pair_y_axis_label <- function(x) {
  x <- pair_axis_label(x)
  x[x == "B"] <- "B cell"
  x[x == "T"] <- "T cell"
  x
}

make_pair_panel <- function(gid) {
  d         <- related_corr[group_id == gid]
  members   <- unique(d$assigned_cell_type)
  member_lv <- d$LV[match(members, d$assigned_cell_type)]
  d         <- d[cell_type %in% members]

  short_title <- short_ct(ct_label(members))
  short_axis  <- pair_axis_label(members)
  row_levels  <- wrap_label(pair_y_axis_label(members), width = 10)
  # Column label = assigned cell type over its LV, stacked on two lines so
  # no rotation is needed in this narrow block.
  col_levels <- paste0(wrap_label(short_axis, width = 10), "\n", member_lv)

  d[, row_label := factor(wrap_label(pair_y_axis_label(cell_type), width = 10),
                          levels = row_levels)]
  d[, col_label := factor(paste0(wrap_label(pair_axis_label(assigned_cell_type), width = 10),
                                 "\n", LV),
                          levels = col_levels)]

  panel_title <- paste(strwrap(paste(short_title, collapse = " vs. "), width = 16), collapse = "\n")

  ggplot(d, aes(x = col_label, y = row_label, fill = cor)) +
    geom_tile(colour = "white", linewidth = 0.3) +
    geom_text(aes(label = sprintf("%.2f", cor), colour = abs(cor) >= 0.8),
              size = pt_mm(FS_HEAT_VALUE), show.legend = FALSE) +
    scale_fill_gradientn(colours = DIVERGING_COLORS, values = DIVERGING_VALUES,
                         limits = c(-1, 1), name = "Pearson r",
                         guide = guide_colourbar(barwidth = unit(20, "mm"),
                                                 barheight = unit(1.8, "mm"),
                                                 title.position = "left",
                                                 title.vjust = 1)) +
    scale_colour_manual(values = TILE_TEXT, guide = "none") +
    scale_x_discrete(expand = c(0, 0)) +
    scale_y_discrete(expand = c(0, 0)) +
    labs(x = NULL, y = NULL, title = panel_title) +
    theme_nature_heatmap(x_angle = 0, label_size = FS_HEAT_LABEL_H) +
    theme(plot.title = element_text(size = FS_SUBTITLE, face = "bold", hjust = 0.5,
                                    lineheight = 0.95, margin = margin(b = 1)))
}

E_panels <- lapply(E_GROUP_ORDER, make_pair_panel)
E_panels[[1]] <- add_tag(E_panels[[1]], "E")

# 3 rows x 2 columns matches the tall, narrow slot E occupies next to D.
panel_E <- wrap_plots(E_panels, ncol = 2)

options(repr.plot.width = 2.2, repr.plot.height = 2.5)
print(panel_E)


## Panel F: RNA-Seq robustness to gene subsampling (ARI)

In [ ]:
gene_frac_ari         <- fread(snakemake@input[["gene_fraction_ari_data"]])
gene_frac_comparisons <- fread(snakemake@input[["gene_fraction_ari_comparisons"]])
stopifnot(all(c("fraction", "ari") %in% names(gene_frac_ari)))
stopifnot(all(c("a", "b", "q") %in% names(gene_frac_comparisons)))

FRACTION_LEVELS <- c("100%", "75%", "50%", "25%", "10%", "5%", "1%")
# The axis title already says "(%)", so the tick labels drop the sign and
# fit horizontally in this narrow column.
FRACTION_TICKS  <- sub("%$", "", FRACTION_LEVELS)
gene_frac_ari[, fraction := factor(fraction, levels = FRACTION_LEVELS)]
stopifnot(!anyNA(gene_frac_ari$fraction))

FRACTION_COLORS <- unlist(cfg$RNASEQ_FRACTION_COLORS)[FRACTION_LEVELS]

mean_df_genefrac <- gene_frac_ari[, .(mean_ari = mean(ari, na.rm = TRUE),
                                      max_ari  = max(ari, na.rm = TRUE)), by = fraction]

# Every comparison in the upstream table is anchored at the full gene set,
# so drawing all six stacks six nested brackets over the data. Four
# pre-specified contrasts are shown -- the no-loss boundary (75%), the
# first significant drop (50%) and the two extremes -- all read unchanged
# from the same adjusted-P table; the full table stays in the source CSV.
GENEFRAC_SHOWN <- c("75%", "50%", "10%", "1%")
comp_genefrac  <- gene_frac_comparisons[a == "100%" & b %in% GENEFRAC_SHOWN]
stopifnot(nrow(comp_genefrac) == length(GENEFRAC_SHOWN))

xpos_genefrac  <- setNames(seq_along(FRACTION_LEVELS), FRACTION_LEVELS)
comp_genefrac  <- assign_bracket_tiers(comp_genefrac, xpos_genefrac)

y_base_genefrac <- max(mean_df_genefrac$max_ari) + 0.04
y_step_genefrac <- 0.090
h_genefrac      <- 0.012
y_max_genefrac  <- y_base_genefrac + (max(comp_genefrac$tier) - 1) * y_step_genefrac +
                   h_genefrac + 0.085

panel_F <- ggplot(gene_frac_ari, aes(fraction, ari, fill = fraction)) +
  geom_boxplot(width = 0.55, outlier.shape = NA, colour = "black",
               linewidth = 0.25, alpha = 0.85) +
  geom_jitter(width = 0.10, size = 0.3, shape = 21, fill = "white",
              colour = "#333333", stroke = 0.12, alpha = 0.75) +
  geom_point(data = mean_df_genefrac, aes(x = fraction, y = mean_ari), shape = 23,
             size = 1.1, fill = "white", colour = "black", stroke = 0.3,
             inherit.aes = FALSE) +
  scale_fill_manual(values = FRACTION_COLORS, na.value = "grey70") +
  scale_x_discrete(labels = setNames(FRACTION_TICKS, FRACTION_LEVELS)) +
  scale_y_continuous(breaks = seq(0, 1, 0.25),
                     labels = c("0", "0.25", "0.5", "0.75", "1.0"),
                     expand = expansion(mult = c(0.02, 0.02))) +
  coord_cartesian(ylim = c(0, y_max_genefrac), clip = "on") +
  labs(x = "Genes used for RNA-Seq (%)", y = "Adjusted Rand Index (ARI)") +
  theme_nature_methods(grid = "none") +
  theme(legend.position = "none",
        axis.text.x = element_text(angle = 60, hjust = 1, vjust = 1, size = FS_LEGEND_TITLE),
        axis.title.y = element_text(angle = 90, margin = margin(r = 0.3)),
        plot.margin = margin(1, 1, 1, 0.3, "mm"))

panel_F <- add_brackets(panel_F, comp_genefrac,
                        y_base = y_base_genefrac, y_step = y_step_genefrac, h = h_genefrac,
                        size = pt_mm(FS_HEAT_LABEL_H))
panel_F <- add_overlay_tag(panel_F, "F")

options(repr.plot.width = 3.6, repr.plot.height = 2.2)
print(panel_F)


## Panel G: GTEx method-level tissue-clustering benchmark (ARI)

In [ ]:
ari_data        <- fread(snakemake@input[["ari_data"]])
ari_comparisons <- fread(snakemake@input[["ari_comparisons"]])
stopifnot(all(c("method", "ari") %in% names(ari_data)))
stopifnot(all(c("a", "b", "q") %in% names(ari_comparisons)))

# Same display name as panels A and B, so the shared colour vector applies.
ari_data[, method := fifelse(method == "GenomicSuperSignature", "GSSig", method)]
ari_comparisons[, a := fifelse(a == "GenomicSuperSignature", "GSSig", a)]
ari_comparisons[, b := fifelse(b == "GenomicSuperSignature", "GSSig", b)]

# Match the executed clustering analysis: four near-tied leading methods are
# pinned in its declared order, followed by descending mean ARI.
ari_method_means <- ari_data[, .(m = mean(ari, na.rm = TRUE)), by = method]
G_PINNED_ORDER <- c("CLAMPfull", "PLIER", "CLAMPbase", "NMF")
G_PINNED_ORDER <- G_PINNED_ORDER[G_PINNED_ORDER %in% ari_method_means$method]
method_order_ari <- c(
  G_PINNED_ORDER,
  ari_method_means[!method %in% G_PINNED_ORDER][order(-m), as.character(method)]
)
ari_data[, method := factor(method, levels = method_order_ari)]

METHOD_COLORS_ARI <- MODEL_COLORS_RAW[method_order_ari]
METHOD_COLORS_ARI[is.na(METHOD_COLORS_ARI)] <- "grey70"
names(METHOD_COLORS_ARI) <- method_order_ari

mean_df_ari <- ari_data[, .(mean_ari = mean(ari, na.rm = TRUE),
                            max_ari  = max(ari, na.rm = TRUE)), by = method]
mean_df_ari[, method := factor(method, levels = method_order_ari)]

# Draw all six pre-specified comparisons exported by the executed clustering
# notebook, with adjusted P values unchanged from the upstream table.
comp_ari <- copy(ari_comparisons)
xpos_ari <- setNames(seq_along(method_order_ari), method_order_ari)
comp_ari <- assign_bracket_tiers(comp_ari, xpos_ari)

y_base_ari <- max(mean_df_ari$max_ari) + 0.05
y_step_ari <- 0.080
h_ari      <- 0.015
y_max_ari  <- y_base_ari + (max(comp_ari$tier) - 1) * y_step_ari + h_ari + 0.085

panel_G <- ggplot(ari_data, aes(method, ari, fill = method)) +
  geom_boxplot(width = 0.55, outlier.shape = NA, colour = "black",
               linewidth = 0.25, alpha = 0.85) +
  geom_jitter(width = 0.10, size = 0.3, shape = 21, fill = "white",
              colour = "#333333", stroke = 0.12, alpha = 0.75) +
  geom_point(data = mean_df_ari, aes(x = method, y = mean_ari), shape = 23,
             size = 1.1, fill = "white", colour = "black", stroke = 0.3,
             inherit.aes = FALSE) +
  scale_fill_manual(values = METHOD_COLORS_ARI, na.value = "grey70") +
  scale_y_continuous(breaks = seq(0, 1, 0.25),
                     labels = c("0", "0.25", "0.5", "0.75", "1.0"),
                     expand = expansion(mult = c(0.02, 0.02))) +
  coord_cartesian(ylim = c(0, y_max_ari), clip = "on") +
  labs(x = NULL, y = "Adjusted Rand Index (ARI)") +
  theme_nature_methods(grid = "none") +
  theme(legend.position = "none",
        axis.text.x = element_text(angle = 35, hjust = 1, vjust = 1, size = FS_AXIS_TEXT),
        axis.title.y = element_text(angle = 90, margin = margin(r = 0.3)),
        plot.margin = margin(1, 1, 1, 0.3, "mm"))

panel_G <- add_brackets(panel_G, comp_ari,
                        y_base = y_base_ari, y_step = y_step_ari, h = h_ari,
                        size = pt_mm(FS_HEAT_LABEL_H))
panel_G <- add_overlay_tag(panel_G, "G")

options(repr.plot.width = 3.6, repr.plot.height = 2.2)
print(panel_G)


## Panel H: subtissue-level B-matrix concordance, with Z-matrix confirmation (`02_b_matrix.ipynb` / `03_global_alignment.ipynb`)

In [ ]:
tissue_subtissue_heatmap <- fread(snakemake@input[["tissue_subtissue_heatmap"]])
z_matrix_subtissue       <- fread(snakemake@input[["z_matrix_subtissue"]])
stopifnot(all(c("Tissue", "Predicted_Tissue", "Pct") %in% names(tissue_subtissue_heatmap)))

hm <- copy(tissue_subtissue_heatmap)

# All 49 subtissues kept, original (alphabetical) order preserved on both
# axes; only the label text is shortened. Abbreviations are listed in the
# figure caption.
hm_order  <- sort(unique(hm$Tissue))
hm_short  <- shorten_gtex(hm_order)
N_SUBTISSUES <- length(hm_order)
stopifnot(!anyDuplicated(hm_short))

# Subtissue names on both axes, compacted by shorten_gtex().
hm[, row_lab := factor(shorten_gtex(Tissue), levels = rev(hm_short))]
hm[, col_lab := factor(shorten_gtex(Predicted_Tissue), levels = hm_short)]

# 2,401 cells: printing all of them is unreadable. Label at most the strongest
# off-diagonal confusion per true-tissue row, and only when it is >= 20%.
# The dense diagonal remains explicit through fill plus black/red outlines.
# Dense numeric annotations use the dedicated compact size to avoid overlap.
SUBTISSUE_LABEL_MIN <- 20
hm[, is_diagonal := Tissue == Predicted_Tissue]
hm[, offdiag_rank := frank(fifelse(is_diagonal, Inf, -Pct), ties.method = "first"),
   by = Tissue]
hm[, show_label := !is_diagonal & Pct >= SUBTISSUE_LABEL_MIN & offdiag_rank == 1]
hm_labels <- hm[show_label == TRUE][order(-Pct)]

diag_cells <- hm[Tissue == Predicted_Tissue]
diag_cells <- merge(diag_cells, z_matrix_subtissue[, .(Tissue, tissue_correct)],
                    by.x = "Predicted_Tissue", by.y = "Tissue", all.x = TRUE)
stopifnot(!anyNA(diag_cells$tissue_correct))

plot_H_core <- ggplot(hm, aes(x = col_lab, y = row_lab, fill = Pct)) +
  geom_tile(colour = "#f0f0f0", linewidth = 0.1) +
  geom_tile(data = diag_cells[tissue_correct == TRUE],
            aes(x = col_lab, y = row_lab), fill = NA, colour = "black",
            linewidth = 0.35, inherit.aes = FALSE) +
  geom_tile(data = diag_cells[tissue_correct == FALSE],
            aes(x = col_lab, y = row_lab), fill = NA, colour = "red",
            linetype = "22", linewidth = 0.35, inherit.aes = FALSE) +
  geom_text(data = hm_labels,
            aes(label = sprintf("%.0f", Pct), colour = Pct >= 65),
            size = pt_mm(FS_DENSE_VALUE), check_overlap = TRUE, show.legend = FALSE) +
  scale_fill_gradientn(colours = GREEN_SCALE, limits = c(0, 100),
                       name = "% of pooled samples (column-normalised)",
                       guide = guide_colourbar(barwidth = unit(24, "mm"),
                                               barheight = unit(1.8, "mm"),
                                               title.position = "left",
                                               title.vjust = 1)) +
  scale_colour_manual(values = TILE_TEXT, guide = "none") +
  scale_x_discrete(expand = c(0, 0)) +
  scale_y_discrete(expand = c(0, 0)) +
  labs(x = "Predicted GTEx subtissue", y = "GTEx subtissue") +
  theme_nature_heatmap(x_angle = 45, label_size = FS_HEAT_LABEL_H) +
  # Legend under the matrix: this panel now sits in a narrow right-hand
  # column, so width is the scarce dimension and height is not.
  theme(legend.position = "bottom", legend.direction = "horizontal")

# wrap_elements() makes the block opaque to patchwork's axis alignment.
# Without it, H's 49 subtissue names set the panel offset for every other
# row on the page and indent them all by ~20 mm; free() would fix that too
# but stops plot_layout() heights from being honoured.
panel_H <- add_tag(wrap_elements(full = plot_H_core), "H")

options(repr.plot.width = 7.2, repr.plot.height = 4.5)
print(panel_H)


## Assembly

Four rows, composed with `patchwork` using **relative numeric ratios only** - no
`unit(..., "mm")` inside the layout and no row height derived from a category count.
The physical canvas is set once, in `ggsave()`:

| Row | Content | Split |
|---|---|---|
| 1 | **A** benchmark boxplots (2 x 2 tissues) + **B** bootstrap win rate | widths `c(3.2, 1)` |
| 2 | **C** assignment-purity heatmaps | full width |
| 3 | **D** LV x cell-type correlations + **E** difficult pairs | widths `c(2.05, 1)` |
| 4 | **F** over **G** in a narrow left column, **H** spanning both on the right | widths `c(1, 2.35)` |

`coord_fixed()` is deliberately absent from every heatmap: fixing the aspect ratio is what
previously forced the page past 400 mm and left wide bands of empty margin around the
fixed-aspect panels. Cells are allowed to be slightly rectangular instead.

Within C and D each sub-row uses the available width with category-count-proportional
panel widths. A compact 1.35:1 row ratio enlarges the lower-row datasets while leaving
the 14-category upper-row heatmaps readable.

Legends are collected per logical group: C carries the purity scale (parked with
`guide_area()` in the gap the shorter bottom sub-row leaves, rather than on a row of its
own), D and E share one correlation scale, H carries its own percentage scale; A, B, F and G
label their categories directly and need none.

Panel H is wrapped in `wrap_elements()` so its 49 subtissue names stay opaque to patchwork's
axis alignment - otherwise every other row on the page is indented to match them.

In [ ]:
# ============================================================
# Physical canvas -- set here and nowhere else
# ------------------------------------------------------------
# 183 mm is Nature's double-column width. The height is a chosen
# page height, not a number summed up from per-panel category
# counts, and both must stay inside one A4 page.
# ============================================================
FIG_W   <- 183
FIG_H   <- 285
A4_W    <- 210
A4_H    <- 297
stopifnot(FIG_W <= A4_W, FIG_H <= A4_H)

# Row height ratios (relative, not millimetres), tuned against the rendered
# page. H is by far the densest panel (49 x 49) and takes the largest
# single-row share; C and DE need enough height for a 14-category heatmap
# plus its title and axis-label band.
# At FIG_H = 285 mm these correspond to roughly 47 / 70 / 58 / 110 mm.
ROW_HEIGHTS <- c(AB = 1.00, C = 1.49, DE = 1.23, FGH = 2.34)

build_supp1 <- function() {
  # wrap_plots() rather than `|` / `/`: those operators flatten an operand
  # that is itself a patchwork into the parent level, which would dissolve
  # these blocks into one giant grid. wrap_plots() nests them as intended.
  row_AB <- wrap_plots(panel_A, panel_B,
                       ncol = 2, widths = c(3.2, 1))

  row_DE <- wrap_plots(panel_D, panel_E,
                       ncol = 2, widths = c(1.9, 1.1), guides = "collect") &
    theme(legend.position = "bottom", legend.direction = "horizontal",
          legend.justification = "right")

  # Bottom block: F over G in a narrow left column, H spanning both of
  # those rows on the right. H is the panel that needs height most, and
  # this is the only arrangement on one page that gives it enough.
  row_FGH <- wrap_plots(
    panel_F,
    panel_G,
    panel_H,
    design = c(area(1, 1, 1, 1),   # F  top-left
               area(2, 1, 2, 1),   # G  bottom-left
               area(1, 2, 2, 2)),  # H  full height, right
    widths  = c(1, 2.35),
    heights = c(1, 1.15))

  wrap_plots(row_AB,
             panel_C,
             row_DE,
             row_FGH,
             ncol = 1, heights = as.numeric(ROW_HEIGHTS))
}

supp1 <- build_supp1()

# ---- Export (the only place millimetres appear) ----
ggsave(snakemake@output[["pdf"]], supp1,
       width = FIG_W, height = FIG_H, units = "mm",
       device = cairo_pdf, bg = "white")

ggsave(snakemake@output[["svg"]], supp1,
       width = FIG_W, height = FIG_H, units = "mm",
       device = svglite::svglite, bg = "white")

ggsave(snakemake@output[["png"]], supp1,
       width = FIG_W, height = FIG_H, units = "mm",
       dpi = 600, device = ragg::agg_png, bg = "white")

out_dir <- dirname(snakemake@output[["pdf"]])
cat(sprintf("supp1: %.0f x %.0f mm (%s A4 %.0f x %.0f mm)\n",
            FIG_W, FIG_H,
            if (FIG_W <= A4_W && FIG_H <= A4_H) "fits within" else "EXCEEDS",
            A4_W, A4_H))
cat("output directory:", out_dir, "\n")
cat("  ", basename(snakemake@output[["pdf"]]), "\n",
    "  ", basename(snakemake@output[["svg"]]), "\n",
    "  ", basename(snakemake@output[["png"]]), " (600 dpi)\n", sep = "")
